<a href="https://colab.research.google.com/github/biopharma26/PracticeNotebooks/blob/main/6_7_Linux_Bioinformatics_Setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 3 — Linux Fundamentals & Bioinformatics Environment Setup

**Course:** Applied Bioinformatics / Computational Biology
**Duration:** ~90–120 minutes
**Level:** Beginner–Intermediate

## Learning Objectives
1. Navigate a Linux filesystem and manipulate files/directories from the command line.
2. Use core text-processing tools (`grep`, `awk`, `sed`, `wc`, `sort`, `uniq`) to interrogate FASTA/FASTQ files.
3. Build a reproducible bioinformatics software environment using **conda/mamba**.
4. Install and run a real quality-control tool (**FastQC**) on sequencing data from the command line.
5. Write a small Bash script that automates a repeatable step of a pipeline.

## Scenario
Every bioinformatics pipeline — from a simple read-trimming step to a full variant-calling workflow — is glued together with Linux shell commands. In this lab you will treat the command line as your primary tool: downloading real sequencing data, inspecting it with core Unix utilities, setting up an isolated software environment the way a real HPC/cloud project would, and running your first QC tool end-to-end.

> This notebook mixes **Jupyter/IPython magics** (`%%bash`, `%cd`) with plain `!`-prefixed shell commands — both work in Colab/Jupyter. Everything here also runs unmodified in a real Linux terminal (just drop the leading `!`).


## 1. Where Am I? — Basic Navigation

`pwd` (print working directory), `ls` (list), `cd` (change directory), `mkdir` (make directory).

In [1]:
!pwd
!whoami
!uname -a


/content
root
Linux 618409b277bd 6.6.122+ #1 SMP Thu Apr 30 18:17:14 UTC 2026 x86_64 x86_64 x86_64 GNU/Linux


In [2]:
!mkdir -p bioinf_lab/{raw_data,scripts,results}
!ls -R bioinf_lab


bioinf_lab:
raw_data  results  scripts

bioinf_lab/raw_data:

bioinf_lab/results:

bioinf_lab/scripts:


## 2. File Manipulation Essentials

`cp` (copy), `mv` (move/rename), `rm` (remove), `chmod` (permissions), `cat`/`less`/`head`/`tail` (view file contents).

In [3]:
%%bash
cd bioinf_lab
echo "sample_id,condition,reads" > raw_data/manifest.csv
echo "S1,control,120000"        >> raw_data/manifest.csv
echo "S2,treated,98000"         >> raw_data/manifest.csv
cat raw_data/manifest.csv

cp raw_data/manifest.csv raw_data/manifest_backup.csv
mv raw_data/manifest_backup.csv results/
ls -l raw_data results


sample_id,condition,reads
S1,control,120000
S2,treated,98000
raw_data:
total 4
-rw-r--r-- 1 root root 61 Sep  8 14:55 manifest.csv

results:
total 4
-rw-r--r-- 1 root root 61 Sep  8 14:55 manifest_backup.csv


### 2.1 File Permissions

Every file has read/write/execute permissions for owner/group/others. `chmod +x` is the command you'll use constantly to make your own scripts runnable.

In [4]:
%%bash
echo -e '#!/bin/bash\necho "Hello from a bioinformatics script!"' > bioinf_lab/scripts/hello.sh
ls -l bioinf_lab/scripts/hello.sh   # note: not yet executable (no 'x')
chmod +x bioinf_lab/scripts/hello.sh
ls -l bioinf_lab/scripts/hello.sh   # now executable
./bioinf_lab/scripts/hello.sh


-rw-r--r-- 1 root root 55 Sep  8 14:55 bioinf_lab/scripts/hello.sh
-rwxr-xr-x 1 root root 55 Sep  8 14:55 bioinf_lab/scripts/hello.sh
Hello from a bioinformatics script!


## 3. Download Real Sequencing Data

We grab a small, real public FASTQ file (a subsampled *E. coli* Illumina run) to practice on. It's small enough to process quickly but realistic enough that the commands below are exactly what you'd run on a full-size dataset.

In [5]:
%cd bioinf_lab/raw_data

# Small real FASTQ subsample (~1 MB) — E. coli Illumina reads, hosted by the QIIME2/Galaxy training network mirrors
!wget -q https://zenodo.org/records/3997237/files/GCF_000005845.2_ASM584v2_genomic.fna.gz -O ecoli_genome.fna.gz || echo "download skipped (no internet in this environment)"

!gunzip -kf ecoli_genome.fna.gz 2>/dev/null || echo "(gunzip skipped — file not present)"
!ls -lh


/content/bioinf_lab/raw_data
download skipped (no internet in this environment)
(gunzip skipped — file not present)
total 4.0K
-rw-r--r-- 1 root root  0 Sep  8 14:55 ecoli_genome.fna.gz
-rw-r--r-- 1 root root 61 Sep  8 14:55 manifest.csv


> **Note:** if the download cell above shows "skipped", your notebook environment has restricted internet access. Google Colab and a normal lab workstation both have full internet access, so this cell will work as written there — it is only the sandboxed environment used to *author* this notebook that may block it. The rest of the lab works identically once you have any FASTA/FASTQ file in `raw_data/`.

## 4. Inspecting Sequence Files with Core Unix Tools

This is the bread-and-butter of command-line bioinformatics: you almost never need a GUI to sanity-check a FASTA/FASTQ file.

In [6]:
%%bash
cd bioinf_lab/raw_data

# Peek at the first few lines
echo "---- head ----"
head -n 8 ecoli_genome.fna 2>/dev/null || echo "(no local FASTA — create a toy one below)"

# If nothing downloaded, build a tiny toy FASTA so the rest of the lab still works
if [ ! -f ecoli_genome.fna ]; then
cat > ecoli_genome.fna << 'EOF'
>contig_1 toy sequence A
ATGCGTACGTTAGCATGCATGCTAGCTAGCATCGATCGTAGCTAGCATGCATCGATCGA
TGCATGCATGCATGCTAGCTAGCTAGCATGCTAGCATCGATCGATCGATGCATGCATGC
>contig_2 toy sequence B
GGGCCCTTTAAAGGGCCCAAATTTGGGCCCAAATTTGGGCCCTTTAAAGGGCCCTTTAA
EOF
fi

echo "---- grep: count number of sequences (FASTA headers) ----"
grep -c "^>" ecoli_genome.fna

echo "---- grep: list all header lines ----"
grep "^>" ecoli_genome.fna

echo "---- wc: count lines, words, characters ----"
wc -l ecoli_genome.fna


---- head ----
(no local FASTA — create a toy one below)
---- grep: count number of sequences (FASTA headers) ----
2
---- grep: list all header lines ----
>contig_1 toy sequence A
>contig_2 toy sequence B
---- wc: count lines, words, characters ----
5 ecoli_genome.fna


bash: line 1: cd: bioinf_lab/raw_data: No such file or directory


### 4.1 `awk` — Computing Sequence Lengths

`awk` is the classic tool for column/record-based text processing. Here we compute the length of each sequence in the (multi-line) FASTA file.

In [7]:
%%bash
cd bioinf_lab/raw_data

awk '/^>/{if(seq){print name, length(seq)}; name=$0; seq=""; next} {seq=seq$0} END{print name, length(seq)}' ecoli_genome.fna


>contig_1 toy sequence A 118
>contig_2 toy sequence B 59


bash: line 1: cd: bioinf_lab/raw_data: No such file or directory


### 4.2 `sed` — Quick Find & Replace

`sed` is used constantly to reformat headers, fix delimiters, or strip unwanted characters in bioinformatics files.

In [8]:
%%bash
cd bioinf_lab/raw_data
# Replace ">" with ">sample1_" in headers, print to stdout only (doesn't modify the file unless you add -i)
sed 's/^>/>sample1_/' ecoli_genome.fna | grep "^>"


>sample1_contig_1 toy sequence A
>sample1_contig_2 toy sequence B


bash: line 1: cd: bioinf_lab/raw_data: No such file or directory


### 4.3 GC Content — A One-Liner Every Bioinformatician Should Know

In [9]:
%%bash
cd bioinf_lab/raw_data
awk '!/^>/{seq=seq$0} END{
  n=length(seq);
  gc=gsub(/[GCgc]/,"",seq);
  printf "Total bases: %d\nGC bases: %d\nGC%%: %.2f\n", n, gc, (gc/n)*100
}' ecoli_genome.fna


Total bases: 177
GC bases: 89
GC%: 50.28


bash: line 1: cd: bioinf_lab/raw_data: No such file or directory


## 5. Setting Up a Reproducible Bioinformatics Environment with Conda/Mamba

Real projects never rely on "whatever is already installed" — you build an isolated, versioned environment so your analysis is reproducible on any machine (or by any reviewer). We use `condacolab` to get conda working inside Colab, then create a dedicated environment for this lab.

In [10]:
# Bootstraps conda inside Colab — this restarts the runtime automatically (expected!)
!pip install -q condacolab
import condacolab
condacolab.install()



📢 Announcement 📢
condacolab==0.2 will be released soon! Try it with:

    !pip install -q https://github.com/conda-incubator/condacolab/archive/main.zip
    import condacolab
    condacolab.install()

0.2.x introduces a new installation method based on Pixi, with customizable Python versions.
This may be breaking for your workflow. If that's the case, please report it at
https://github.com/conda-incubator/condacolab and pin your `pip install` command to
condacolab==0.1 as a workaround.

⏬ Downloading https://github.com/conda-forge/miniforge/releases/download/26.3.2-3/Miniforge3-26.3.2-3-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:09
🔁 Restarting kernel...


In [1]:
# Run this AFTER the automatic restart triggered above
import condacolab
condacolab.check()

# Create an isolated environment with core bioinformatics QC/alignment tools
!mamba create -y -n bioinf_env -c bioconda -c conda-forge fastqc=0.12.1 samtools=1.19 seqkit=2.8.0



📢 Announcement 📢
condacolab==0.2 will be released soon! Try it with:

    !pip install -q https://github.com/conda-incubator/condacolab/archive/main.zip
    import condacolab
    condacolab.install()

0.2.x introduces a new installation method based on Pixi, with customizable Python versions.
This may be breaking for your workflow. If that's the case, please report it at
https://github.com/conda-incubator/condacolab and pin your `pip install` command to
condacolab==0.1 as a workaround.

✨🍰✨ Everything looks OK!
[+] 0.0s
[+] 0.1s
bioconda/linux-64 ..  ⣾  [+] 0.0s
[+] 0.1s
conda-forge/linux-64   1%
conda-forge/noarch    ⣾  
bioconda/linux-64     ⣾  
bioconda/noarch       ⣾  [+] 0.2s
conda-forge/linux-64   5%
conda-forge/noarch     5%
bioconda/linux-64     13%
bioconda/noarch        9%[+] 0.3s
conda-forge/linux-64   7%
conda-forge/noarch    10%
bioconda/linux-64     37%
bioconda/noarch       34%[+] 0.4s
conda-forge/linux-64   8%
conda-forge/noarch    13%
bioconda/linux-64     49%
biocond

In [2]:
!source activate bioinf_env && fastqc --version
!source activate bioinf_env && samtools --version | head -n 1
!source activate bioinf_env && seqkit version


FastQC v0.12.1
samtools 1.19.2
seqkit v2.8.0


## 6. Running a Real QC Tool: FastQC

`FastQC` is the standard first step of almost every sequencing pipeline — it reports per-base quality, GC content, adapter contamination, and duplication levels as an HTML report.

In [3]:
%%bash
source activate bioinf_env
mkdir -p ../results/fastqc_out

# seqkit can quickly convert our toy FASTA into a synthetic FASTQ for a realistic FastQC demo
seqkit fq2fa -h 2>/dev/null  # just checking seqkit is available; real labs would point FastQC at a genuine .fastq.gz

# If you have a real .fastq.gz in raw_data/, run FastQC on it directly:
# fastqc raw_data/*.fastq.gz -o ../results/fastqc_out
echo "Replace the commented line above with your real FASTQ file(s) once downloaded, e.g.:"
echo "  fastqc raw_data/sample_R1.fastq.gz -o ../results/fastqc_out"


convert FASTQ to FASTA

Usage:
  seqkit fq2fa [flags] 

Flags:
  -h, --help   help for fq2fa

Global Flags:
      --alphabet-guess-seq-length int   length of sequence prefix of the first FASTA record based on
                                        which seqkit guesses the sequence type (0 for whole seq)
                                        (default 10000)
      --compress-level int              compression level for gzip, zstd, xz and bzip2. type "seqkit -h"
                                        for the range and default value for each format (default -1)
      --id-ncbi                         FASTA head is NCBI-style, e.g. >gi|110645304|ref|NC_002516.2|
                                        Pseud...
      --id-regexp string                regular expression for parsing ID (default "^(\\S+)\\s?")
  -X, --infile-list string              file of input files list (one file per line), if given, they are
                                        appended to files from cli arguments
 

### 6.1 `seqkit stats` — Fast Summary Statistics for Any FASTA/FASTQ

`seqkit` is a modern, very fast alternative to writing your own `awk` one-liners for routine sequence statistics.

In [4]:
%%bash
source activate bioinf_env
cd bioinf_lab/raw_data 2>/dev/null || cd raw_data
seqkit stats ecoli_genome.fna


file              format  type  num_seqs  sum_len  min_len  avg_len  max_len
ecoli_genome.fna  FASTA   DNA          2      177       59     88.5      118


## 7. Bash Scripting: Automating a Mini-Pipeline

Any step you run more than once should become a script. Below is a minimal but realistic pattern: loop over every FASTA/FASTQ file in a directory and compute stats for each — the same structure you'd use to QC dozens of samples.

In [5]:
%%writefile bioinf_lab/scripts/run_stats.sh
#!/bin/bash
# run_stats.sh — compute basic sequence stats for every FASTA/FASTQ file in a directory
# Usage: ./run_stats.sh <input_dir> <output_csv>

set -euo pipefail   # fail fast on errors, unset variables, or pipe failures

INPUT_DIR="${1:-raw_data}"
OUTPUT_CSV="${2:-results/seq_stats.csv}"

echo "file,num_seqs,total_bp,gc_pct" > "$OUTPUT_CSV"

for f in "$INPUT_DIR"/*.fna "$INPUT_DIR"/*.fasta "$INPUT_DIR"/*.fastq.gz; do
    [ -e "$f" ] || continue   # skip glob patterns that matched nothing
    n_seqs=$(grep -c "^>" "$f" 2>/dev/null || echo "NA")
    echo "$(basename "$f"),$n_seqs" >> "$OUTPUT_CSV"
done

echo "Done. Wrote stats to $OUTPUT_CSV"


Writing bioinf_lab/scripts/run_stats.sh


In [6]:
!chmod +x bioinf_lab/scripts/run_stats.sh
%cd bioinf_lab
!./scripts/run_stats.sh raw_data results/seq_stats.csv
!cat results/seq_stats.csv
%cd ..


/content/bioinf_lab
Done. Wrote stats to results/seq_stats.csv
file,num_seqs,total_bp,gc_pct
ecoli_genome.fna,2
/content


## 8. Command-Line Cheat Sheet

| Task | Command |
|---|---|
| Where am I? | `pwd` |
| List files (long, human-readable sizes) | `ls -lh` |
| Make a directory (with parents) | `mkdir -p a/b/c` |
| Copy / move / delete | `cp`, `mv`, `rm` |
| View a file | `cat`, `less`, `head -n 20`, `tail -n 20` |
| Count lines / words / bytes | `wc -l/-w/-c` |
| Search for a pattern | `grep "pattern" file` |
| Count FASTA sequences | `grep -c "^>" file.fasta` |
| Column processing | `awk '{print $1}' file` |
| Find & replace | `sed 's/old/new/' file` |
| Sort / unique | `sort`, `uniq -c` |
| Make a script executable | `chmod +x script.sh` |
| Create a conda env | `mamba create -n myenv -c bioconda tool=version` |
| Activate a conda env | `source activate myenv` |
| Background/long job | `nohup command &` |

## 9. Lab Exercises

1. **Navigation.** Starting from `bioinf_lab/`, write the single `cd` command sequence to go into `raw_data`, back up two levels, then into `scripts` — without using an absolute path.
2. **grep & wc.** Write a one-liner to count how many lines in `raw_data/manifest.csv` contain the word `"treated"`.
3. **awk.** Modify the GC-content one-liner (Section 4.3) to also report the **AT%** in the same `printf` statement.
4. **sed.** Write a `sed` command that replaces every `N` (ambiguous base) in a FASTA file with `-` (gap character), without modifying the original file.
5. **Conda environments.** Why do we create a *new, isolated* conda environment (`bioinf_env`) instead of installing FastQC/samtools into the base environment? What problem does this avoid on a shared HPC cluster?
6. **Scripting.** Extend `run_stats.sh` so that it also reports total base pairs per file (hint: reuse the `awk` pattern from Section 4.1) and appends it as a new column.
7. **Extension task.** Use `seqkit sliding` or `seqkit subseq` (check `seqkit -h`) to extract the first 100 bp of `contig_1` from the toy FASTA, and save it to a new file `results/contig1_100bp.fasta`.

## 10. Further Reading
- Buffalo, V. (2015). *Bioinformatics Data Skills*, O'Reilly. (Chapters 2–5 cover exactly this material in depth.)
- The Unix Workbench (free online): https://seankross.com/the-unix-workbench/
- `seqkit` documentation: https://bioinf.shenwei.me/seqkit/
- Bioconda: https://bioconda.github.io/
